# 01 -- Crawler Research

Scratch space for developing and validating Task 2's USCIS crawler against
the *real* site, one stage at a time, before trusting it to run unattended.

This notebook does not reimplement the crawler -- every function it calls
(`find_form_detail_links`, `find_pdf_links`, `discover_pdf_links`, `run`) is
imported from `src/crawler/uscis_crawler.py` and `src/crawler/base_crawler.py`.
If a bug is found here, the fix belongs in `src/`, not in a notebook cell.

## Validated research result

The USCIS "All Forms" index page does **not** link PDFs directly:

- `https://www.uscis.gov/forms/all-forms` returned HTTP 200 with 383 links,
  **zero** of which were direct `.pdf` links.
- What it *does* link are individual form-detail pages, e.g. `/i-485`,
  `/n-400`, `/i-485supa`.
- Each form-detail page (e.g. `https://www.uscis.gov/i-485`) returned HTTP
  200 and exposed the actual PDFs, e.g.:
  - `https://www.uscis.gov/sites/default/files/document/forms/i-485.pdf`
  - `https://www.uscis.gov/sites/default/files/document/forms/i-485instr.pdf`

This is why the crawler is two-stage: **index page -> form-detail pages ->
PDF links -> download.** The rest of this notebook re-demonstrates each
stage against the live site, on a small sample.

## Project setup and imports

In [1]:
import sys
from pathlib import Path

# Notebooks live in notebooks/, but the reusable pipeline code lives in
# src/ at the project root. Jupyter sets the working directory to wherever
# it was launched from, which isn't reliable -- so we search upward for
# requirements.txt (a stable marker of the project root) instead of
# hardcoding "..".
def find_project_root(marker="requirements.txt"):
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / marker).exists():
            return candidate
    raise RuntimeError("Could not locate project root (no requirements.txt found above cwd)")

PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

PROJECT_ROOT


WindowsPath('d:/USD/Projects/a590/newstart-ai')

In [2]:
from src.crawler import base_crawler, uscis_crawler

base_crawler, uscis_crawler


(<module 'src.crawler.base_crawler' from 'd:\\USD\\Projects\\a590\\newstart-ai\\src\\crawler\\base_crawler.py'>,
 <module 'src.crawler.uscis_crawler' from 'd:\\USD\\Projects\\a590\\newstart-ai\\src\\crawler\\uscis_crawler.py'>)

## Loading the USCIS index URL

`INDEX_URLS` is defined in `uscis_crawler.py`, not here -- this cell only
*displays* the configuration this notebook run will use.

In [3]:
print("BASE_URL:", uscis_crawler.BASE_URL)
print("INDEX_URLS:", uscis_crawler.INDEX_URLS)

index_url = uscis_crawler.INDEX_URLS[0]
index_url


BASE_URL: https://www.uscis.gov
INDEX_URLS: ['https://www.uscis.gov/forms/all-forms']


'https://www.uscis.gov/forms/all-forms'

## Stage 1 -- discovering form-detail links

Calls `find_form_detail_links`, which fetches the index page once and
returns only the links matching the USCIS form-detail path pattern (see
`uscis_crawler.FORM_DETAIL_PATH_RE`) -- navigation pages like
`/forms/filing-guidance` are filtered out, not just "every internal link".

In [4]:
detail_links = uscis_crawler.find_form_detail_links(index_url)
print(f"{len(detail_links)} form-detail page(s) discovered")
detail_links[:10]


102 form-detail page(s) discovered


['https://www.uscis.gov/i-9',
 'https://www.uscis.gov/i-485',
 'https://www.uscis.gov/i-765',
 'https://www.uscis.gov/i-90',
 'https://www.uscis.gov/n-400',
 'https://www.uscis.gov/i-129f',
 'https://www.uscis.gov/i-130',
 'https://www.uscis.gov/i-360',
 'https://www.uscis.gov/i-600',
 'https://www.uscis.gov/i-751']

## Stage 2 -- discovering PDF links on one detail page

Picks the first discovered detail page and looks for direct PDF links on
it -- this is the step that was missing from the original (incorrect)
single-stage assumption.

In [5]:
sample_detail_url = detail_links[0]
sample_detail_url


'https://www.uscis.gov/i-9'

In [6]:
pdf_links = uscis_crawler.find_pdf_links(sample_detail_url)
print(f"{len(pdf_links)} PDF link(s) found on {sample_detail_url}")
pdf_links


4 PDF link(s) found on https://www.uscis.gov/i-9


['https://www.uscis.gov/sites/default/files/document/forms/i-9.pdf',
 'https://www.uscis.gov/sites/default/files/document/forms/i-9instr.pdf',
 'https://www.uscis.gov/sites/default/files/document/forms/i-9-spanish.pdf',
 'https://www.uscis.gov/sites/default/files/document/forms/i9-INS-Spanish.pdf']

## Full discovery pipeline (index -> detail pages -> PDFs)

`discover_pdf_links` runs both stages end-to-end and returns the final,
deduplicated PDF URL list -- **without downloading anything**. This is the
safe way to see what a full crawl would fetch before committing to `run()`.

This will visit every discovered detail page, so it takes a while and
prints progress via `logging` rather than one line per page.

In [9]:
import logging
logging.basicConfig(level=logging.INFO, format="%(message)s")

all_pdf_links = uscis_crawler.discover_pdf_links()
print(f"\nTotal unique PDF links discovered: {len(all_pdf_links)}")
all_pdf_links[:10]


index https://www.uscis.gov/forms/all-forms: 102 form-detail page(s) discovered
index https://www.uscis.gov/forms/all-forms: 102 detail page(s) processed, 0 failed
discovery complete: 281 unique PDF link(s) found



Total unique PDF links discovered: 281


['https://www.uscis.gov/sites/default/files/document/forms/i-9.pdf',
 'https://www.uscis.gov/sites/default/files/document/forms/i-9instr.pdf',
 'https://www.uscis.gov/sites/default/files/document/forms/i-9-spanish.pdf',
 'https://www.uscis.gov/sites/default/files/document/forms/i9-INS-Spanish.pdf',
 'https://www.uscis.gov/sites/default/files/document/forms/i-485.pdf',
 'https://www.uscis.gov/sites/default/files/document/forms/i-485instr.pdf',
 'https://www.together.gov/assets/docs/Ms.%20L%20v.%20ICE%20Settlement.pdf',
 'https://www.uscis.gov/sites/default/files/document/forms/i-765.pdf',
 'https://www.uscis.gov/sites/default/files/document/forms/i-765instr.pdf',
 'https://www.uscis.gov/sites/default/files/document/forms/i-765ws.pdf']

## Small sample download

`run(limit=5)` downloads at most 5 *new* files (any already recorded in
`data/raw/uscis/manifest.csv` from a previous run don't count against the
limit and aren't re-fetched). This is the only cell in this notebook that
writes files or hits `download_pdf`'s validation (domain check + PDF
signature check).

In [10]:
downloaded = uscis_crawler.run(limit=5)
downloaded


index https://www.uscis.gov/forms/all-forms: 102 form-detail page(s) discovered
index https://www.uscis.gov/forms/all-forms: 102 detail page(s) processed, 0 failed
discovery complete: 281 unique PDF link(s) found
download complete: 5 new file(s), 276 already had, 0 failure(s)


[WindowsPath('D:/USD/Projects/a590/newstart-ai/data/raw/uscis/i-9.pdf'),
 WindowsPath('D:/USD/Projects/a590/newstart-ai/data/raw/uscis/i-9instr.pdf'),
 WindowsPath('D:/USD/Projects/a590/newstart-ai/data/raw/uscis/i-9-spanish.pdf'),
 WindowsPath('D:/USD/Projects/a590/newstart-ai/data/raw/uscis/i9-INS-Spanish.pdf'),
 WindowsPath('D:/USD/Projects/a590/newstart-ai/data/raw/uscis/i-485.pdf')]

In [11]:
for path in downloaded:
    print("OK:", path)

manifest_path = uscis_crawler.RAW_DIR / "manifest.csv"
if manifest_path.exists():
    print("\nmanifest.csv tail:")
    print("\n".join(manifest_path.read_text().splitlines()[-6:]))


OK: D:\USD\Projects\a590\newstart-ai\data\raw\uscis\i-9.pdf
OK: D:\USD\Projects\a590\newstart-ai\data\raw\uscis\i-9instr.pdf
OK: D:\USD\Projects\a590\newstart-ai\data\raw\uscis\i-9-spanish.pdf
OK: D:\USD\Projects\a590\newstart-ai\data\raw\uscis\i9-INS-Spanish.pdf
OK: D:\USD\Projects\a590\newstart-ai\data\raw\uscis\i-485.pdf

manifest.csv tail:
filename,source_url,sha256,downloaded_at
i-9.pdf,https://www.uscis.gov/sites/default/files/document/forms/i-9.pdf,780f348c34df694bb0b4dbbfaf9f22b99b9757b80d16a37ba89aadf069597281,2026-07-11T06:48:34.795836+00:00
i-9instr.pdf,https://www.uscis.gov/sites/default/files/document/forms/i-9instr.pdf,c66c3818fbecbfb87b5f0560f4dfd4086905daf72ca29dba15ce4c8d49dc4ca3,2026-07-11T06:48:36.958342+00:00
i-9-spanish.pdf,https://www.uscis.gov/sites/default/files/document/forms/i-9-spanish.pdf,7ebe72cbd3d0cb9a12a659762c22bc9a4928af31721322c3404686ada9774233,2026-07-11T06:48:39.266905+00:00
i9-INS-Spanish.pdf,https://www.uscis.gov/sites/default/files/document/form

## Notes: robots.txt, rate limiting, retries, and responsible crawling

- **robots.txt**: check `https://www.uscis.gov/robots.txt` by hand before
  pointing `INDEX_URLS` at a new section of the site. This project does not
  auto-parse robots.txt; treat it as a manual pre-flight check, not
  something the crawler enforces for you.
- **Rate limiting**: `base_crawler.REQUEST_DELAY_SECONDS` (2s) applies to
  *every* request via `polite_get`, including each detail-page fetch during
  discovery. Don't call `requests` directly from a notebook cell to "speed
  things up" -- that's exactly the drift the shared base module exists to
  prevent.
- **Retries**: not implemented. A transient failure on a detail page is
  logged and skipped by `discover_pdf_links`; a failure on the index page
  itself propagates, since there's nothing to discover without it.
- **Validation on download**: `download_pdf` now checks that the URL (and
  the final, post-redirect response URL) stays on an allowed domain
  (`uscis.gov`), and that the response actually starts with the PDF
  signature (`%PDF`) rather than trusting the `.pdf` extension or
  `Content-Type` header alone.
- **Manifest safety**: re-running this notebook won't re-download files
  already in `manifest.csv`, and won't silently overwrite a different file
  that happens to share a name (see `ManifestWriter.resolve_filename` in
  `base_crawler.py`).
- **Identification**: `USER_AGENT` names this as a USD capstone crawler.
  Don't override it to look like a browser.
- **Scope discipline**: use `discover_pdf_links()` (no download) and small
  `limit=` values liberally while developing. An unlimited `run()` is a
  decision to make deliberately, not a notebook default -- see the
  `__main__` guard in `uscis_crawler.py`, which itself is capped at 5.